In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.12 Kinetic Theory: Collisions, Mean Free Path, and Transport

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.12",
    title="Kinetic Theory: Collisions, Mean Free Path, and Transport",
    blurb="The volume's equilibrium machinery never asked how equilibrium "
    "is enforced. Here the enforcers get their due: sixty-five nanometres "
    "of freedom between collisions, seven billion collisions per second, "
    "an event-driven gas whose free paths land on the predicted "
    "exponential, a viscosity that famously ignores pressure, and the "
    "effusion physics that once separated uranium isotopes.",
    difficulty="intermediate",
    estimate="130–160 min",
)

## Notebook overview

Volume V has treated collisions the way accountants treat their auditors:
essential, and never on stage. The Maxwell–Boltzmann distribution of
[§5.6](ideal-gas-fundamental-relation.ipynb) was derived from counting
alone; the relaxation experiments of
[§5.11](taste-of-nonequilibrium.ipynb) invoked "molecular chaos" and
moved on. This notebook finally puts the collisions themselves on stage,
because they carry their own quantitative physics: *how far* a molecule
flies between encounters (about $65\ \mathrm{nm}$ in the air in front of
you), *how often* it is hit (about seven billion times per second), and —
the payoff — how those two numbers assemble into the *transport
coefficients* that connect microscopic chaos to tabulated,
engineering-handbook properties: diffusivity, viscosity, thermal
conductivity.

The historical stakes were high. Maxwell's kinetic prediction that a
gas's viscosity should *not depend on its pressure* struck his
contemporaries as absurd, so he measured it himself, with his wife
Katherine tending the apparatus, and the absurd prediction held
{cite}`maxwell1867`: doubling the density doubles the carriers but
halves each carrier's reach. We verify his cancellation, run an
event-driven hard-disk gas whose measured free paths land on the
predicted exponential, and close with effusion: the escape of molecules
through a small hole, whose $1/\sqrt m$ selectivity was scaled up, stage
by weary stage, into the isotope-separation plants of the 1940s. The
rigorous version of everything here is Chapman–Enskog theory
{cite}`chapmancowling1970`; the volume's standing reference remains
{cite}`nolting6`.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the
answer is wrong; it means the output did not match what the check
expected, which may be a genuine error, a different-but-valid
convention, or too tight a tolerance. Treat a ✗ as a prompt to locate
the discrepancy. Passing is strong evidence, not proof.

## Theory in brief

**The relative-speed factor.** A molecule's collision rate depends not on
its own speed but on its speed *relative* to the other molecules. For
Maxwell–Boltzmann velocities every Cartesian component of
$\mathbf v_1 - \mathbf v_2$ is Gaussian with twice the single-particle
variance, so the relative speed is Maxwell-distributed with
$\sqrt 2$ times the scale:

```{math}
:label: eq-kt-vrel
\langle v_{\rm rel} \rangle \;=\; \sqrt 2\,\langle v \rangle,
\qquad
\langle v \rangle \;=\; \sqrt{\frac{8 k_B T}{\pi m}},
```

an identity that holds in any dimension and supplies the $\sqrt 2$ that
decorates every mean-free-path formula.

**Mean free path.** Model molecules as hard spheres of diameter $d$: two
collide when their centers approach within $d$, so a moving molecule
sweeps a collision cylinder of cross-section $\sigma = \pi d^2$. At
number density $n$ it suffers $n \sigma \langle v_{\rm rel}\rangle$
collisions per second, and flies, on average,

```{math}
:label: eq-kt-lambda
\lambda \;=\; \frac{\langle v\rangle}{n \sigma \langle v_{\rm rel}
\rangle} \;=\; \frac{1}{\sqrt 2\, n\, \pi d^2}
\;=\; \frac{k_B T}{\sqrt 2\, \pi d^2\, p}
```

between hits. Because the collisions are uncorrelated, the free-path
*distribution* is exponential, $P(\ell) = e^{-\ell/\lambda}/\lambda$: a
prediction our simulation can test directly. One dimensional subtlety
matters for that test: the cross-section is the *set of impact
parameters that produce a hit*, which in three dimensions is a disc of
radius $d$ (area $\pi d^2$) but in a two-dimensional gas of disks is a
*segment* of half-width $d$ — width $2d$, so $\lambda_{\rm 2D} =
1/(2\sqrt2\,n\,d)$. Forgetting the factor $2$ makes the simulation
disagree with theory by a factor $2$, which is exactly how we first
found it.

**Transport, the back-of-envelope way.** A gradient of anything carried
by molecules (momentum, energy, the molecules themselves) relaxes
because carriers from one layer deposit their cargo a distance
$\sim\lambda$ away. The elementary estimate gives all three coefficients
at once:

```{math}
:label: eq-kt-transport
D \;\approx\; \tfrac13 \langle v\rangle \lambda, \qquad
\eta \;\approx\; \tfrac13 n m \langle v\rangle \lambda, \qquad
\kappa \;\approx\; \tfrac13 n c_v \langle v\rangle \lambda,
```

with $c_v$ the heat capacity per molecule. Two consequences outrank the
prefactors. First, Maxwell's shock: since $\lambda \propto 1/n$, the
density cancels in $\eta$ — *viscosity is independent of pressure*.
Second, the honest accuracy: the $\tfrac13$ is a mnemonic, not a
theorem, and lands within a factor of about $1.5$ of reality. The full
Chapman–Enskog solution of the Boltzmann equation
{cite}`chapmancowling1970` replaces it, for hard spheres, by

```{math}
:label: eq-kt-enskog
\eta_{\rm CE} \;=\; \frac{5}{16}\,
\frac{\sqrt{\pi m k_B T}}{\pi d^2},
```

which meets measured noble-gas viscosities at the percent level once
$d$ is chosen consistently.

**Effusion.** Puncture the container with a hole much smaller than
$\lambda$ and molecules escape *ballistically*, one by one, whenever
their thermal flight happens to cross the opening. The escape flux is
the one-sided average of $v_z$ over the Maxwell–Boltzmann distribution,

```{math}
:label: eq-kt-effusion
\Phi \;=\; \tfrac14\, n\, \langle v \rangle ,
```

and because $\langle v\rangle \propto 1/\sqrt m$, a hole is a *mass
filter* (Graham's law): the escaping beam is enriched in the lighter
species by $\sqrt{m_2/m_1}$ per pass. For the uranium hexafluorides
$^{235}\mathrm{UF}_6$ and $^{238}\mathrm{UF}_6$ that factor is a
desperately thin $1.0043$, which is why the wartime gaseous-diffusion
plant at Oak Ridge needed thousands of stages in series. The escaping
beam is also *faster* than the gas it leaves (fast molecules find the
hole more often): its speeds are distributed $\propto v f(v)$, with mean
$\tfrac{3\pi}{16}\sqrt 2\,\langle v\rangle \approx 1.18\,\langle
v\rangle$.

## Setup

SI units and `scipy.constants` for every physical number; the hard-disk
laboratory runs in its own reduced units (box side $1$, mean speed of
order $1$). Randomness (initial velocities, effusion sampling) is
seeded.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy.constants import atm
from scipy.constants import k as KB  # J/K
from scipy.constants import u as AMU  # kg
from scipy.integrate import quad

from ecp import animate, draw, validate

rng = np.random.default_rng(0)


def mean_speed(m, T):
    """Maxwell–Boltzmann mean speed sqrt(8 kB T / (pi m)).

    The <v> of eq-kt-vrel: the average magnitude of a thermal velocity in
    three dimensions.

    Parameters
    ----------
    m : float
        Molecular mass in kg.
    T : float
        Temperature in K.

    Returns
    -------
    float
        Mean speed in m/s.
    """
    return np.sqrt(8.0 * KB * T / (np.pi * m))


def mean_free_path(T, p, d):
    """Hard-sphere mean free path kB T / (sqrt2 pi d^2 p), eq-kt-lambda.

    The average flight distance between collisions for a dilute gas of
    spheres of diameter d at temperature T and pressure p.

    Parameters
    ----------
    T : float
        Temperature in K.
    p : float
        Pressure in Pa.
    d : float
        Hard-sphere (collision) diameter in m.

    Returns
    -------
    float
        Mean free path in m.
    """
    return KB * T / (np.sqrt(2.0) * np.pi * d**2 * p)


def eta_elementary(n, m, T, d):
    """Elementary kinetic viscosity (1/3) n m <v> lambda, eq-kt-transport.

    The mnemonic estimate: momentum carried a mean free path by carriers
    of density n. Its virtue is transparency, not the prefactor.

    Parameters
    ----------
    n : float
        Number density in 1/m^3.
    m : float
        Molecular mass in kg.
    T : float
        Temperature in K.
    d : float
        Hard-sphere diameter in m.

    Returns
    -------
    float
        Shear viscosity in Pa s.
    """
    lam = 1.0 / (np.sqrt(2.0) * n * np.pi * d**2)
    return n * m * mean_speed(m, T) * lam / 3.0


def eta_chapman_enskog(m, T, d):
    """Chapman–Enskog hard-sphere viscosity (5/16) sqrt(pi m kB T)/(pi d^2).

    The first-order solution of the Boltzmann equation, eq-kt-enskog:
    the professional version of the elementary estimate, accurate at the
    percent level for noble gases with a consistent diameter.

    Parameters
    ----------
    m : float
        Molecular mass in kg.
    T : float
        Temperature in K.
    d : float
        Hard-sphere diameter in m.

    Returns
    -------
    float
        Shear viscosity in Pa s.
    """
    return 5.0 / 16.0 * np.sqrt(np.pi * m * KB * T) / (np.pi * d**2)


def hard_disk_gas(N, L, d_disk, n_events, rng, masses=None):
    """Event-driven gas of N elastic hard disks in an L-by-L box.

    Advances the exact dynamics collision by collision: all pair and wall
    collision times are computed in closed form (the pair condition
    |dr + dv t| = d_disk is a quadratic in t), the earliest event is
    executed, and elastic impulses are exchanged along the line of
    centers (mass-weighted, so unequal species thermalise correctly).
    Free-path lengths (speed times time since a disk's previous PAIR
    collision) and per-species kinetic energies are recorded as they
    happen.

    Parameters
    ----------
    N : int
        Number of disks.
    L : float
        Box side; disks are confined to [d/2, L - d/2] in each axis.
    d_disk : float
        Disk diameter (centers collide at separation d_disk).
    n_events : int
        Number of events (pair or wall) to execute.
    rng : numpy.random.Generator
        Seeded generator for the initial velocities.
    masses : numpy.ndarray or None
        Per-disk masses (default all 1).

    Returns
    -------
    dict
        ``free_paths`` (array), ``energy_trace`` (list of per-species
        mean kinetic energies, sampled every 50 events),
        ``snapshots`` (positions sampled every ``max(1, n_events//200)``
        events, for animation), ``positions``, ``velocities``.
    """
    g = int(np.ceil(np.sqrt(N)))
    xs = (np.arange(g) + 0.5) / g * L
    pos = np.array([[xs[i], xs[j]] for i in range(g) for j in range(g)])[:N]
    vel = rng.normal(size=(N, 2))
    vel -= vel.mean(axis=0)
    m = np.ones(N) if masses is None else masses.copy()
    two_species = masses is not None
    if two_species:
        # Start the heavy species genuinely cold: a quarter of the light
        # species' velocity scale gives it 4·(1/4)² = 1/4 of the kinetic
        # energy per particle. (Halving would be a trap: the factor 4 in
        # mass exactly cancels (1/2)², and the two species would START in
        # equipartition — we fell into it, and the figure caught it.)
        vel[m > 1.0] *= 0.25

    t_now = 0.0
    last_pair = np.full(N, np.nan)
    free_paths = []
    energy_trace = []
    snapshots = []
    snap_every = max(1, n_events // 200)
    iu, ju = np.triu_indices(N, 1)

    for event in range(n_events):
        t_wall = np.full((N, 2), np.inf)
        for ax in range(2):
            up = vel[:, ax] > 0
            dn = vel[:, ax] < 0
            t_wall[up, ax] = (L - d_disk / 2 - pos[up, ax]) / vel[up, ax]
            t_wall[dn, ax] = (d_disk / 2 - pos[dn, ax]) / vel[dn, ax]
        t_wall_min = float(t_wall.min())

        dr = pos[iu] - pos[ju]
        dv = vel[iu] - vel[ju]
        b = np.einsum("ij,ij->i", dr, dv)
        a2 = np.einsum("ij,ij->i", dv, dv)
        c = np.einsum("ij,ij->i", dr, dr) - d_disk**2
        disc = b * b - a2 * c
        ok = (b < 0) & (disc > 0) & (a2 > 0)
        t_pair = np.full(len(iu), np.inf)
        t_pair[ok] = (-b[ok] - np.sqrt(disc[ok])) / a2[ok]
        t_pair[t_pair < 0] = np.inf
        k_min = int(np.argmin(t_pair))

        if t_pair[k_min] < t_wall_min:
            dt = float(t_pair[k_min])
            pos += vel * dt
            t_now += dt
            i1, j1 = int(iu[k_min]), int(ju[k_min])
            for q in (i1, j1):
                if np.isfinite(last_pair[q]):
                    free_paths.append(
                        float(np.linalg.norm(vel[q])) * (t_now - last_pair[q])
                    )
                last_pair[q] = t_now
            nvec = pos[i1] - pos[j1]
            nvec /= np.linalg.norm(nvec)
            dvn = float(np.dot(vel[i1] - vel[j1], nvec))
            vel[i1] -= (2.0 * m[j1] / (m[i1] + m[j1])) * dvn * nvec
            vel[j1] += (2.0 * m[i1] / (m[i1] + m[j1])) * dvn * nvec
        else:
            dt = t_wall_min
            pos += vel * dt
            t_now += dt
            w = np.unravel_index(int(np.argmin(t_wall)), t_wall.shape)
            vel[w[0], w[1]] *= -1.0

        if two_species and event % 50 == 0:
            e_light = (
                0.5
                * float((m[m == 1.0, None] * vel[m == 1.0] ** 2).sum())
                / int((m == 1.0).sum())
            )
            e_heavy = (
                0.5
                * float((m[m > 1.0, None] * vel[m > 1.0] ** 2).sum())
                / int((m > 1.0).sum())
            )
            energy_trace.append((e_light, e_heavy))
        if event % snap_every == 0:
            snapshots.append(pos.copy())

    return {
        "free_paths": np.array(free_paths),
        "energy_trace": energy_trace,
        "snapshots": snapshots,
        "positions": pos,
        "velocities": vel,
    }

## Exercise 1 — The √2 that rules the formulas

Everything in {eq}`eq-kt-lambda` hangs on the claim
$\langle v_{\rm rel}\rangle = \sqrt2\,\langle v\rangle$ of
{eq}`eq-kt-vrel`, so it earns the first check. The argument is pure
Gaussian algebra: each component of $\mathbf v_1 - \mathbf v_2$ is the
difference of two independent zero-mean Gaussians of variance
$k_BT/m$, hence Gaussian with variance $2k_BT/m$; the relative velocity
is therefore Maxwell-distributed at *doubled temperature*, and every
speed average inherits a factor $\sqrt2$.

**Part a)** Draw $2\times10^5$ pairs of three-dimensional
Maxwell–Boltzmann velocities at $k_BT/m = 1$ (each component a standard
normal from the seeded generator) and form the ratio of sample means
$\langle|\mathbf v_1 - \mathbf v_2|\rangle / \langle|\mathbf v|\rangle$
with `numpy.linalg.norm`. Verify it lands on $\sqrt 2$ within `rtol=1e-2`
(the residual is Monte Carlo noise).

**Part b)** Verify the absolute scale: the sample $\langle|\mathbf v|
\rangle$ against the closed form $\sqrt{8/\pi}$ (in these units) at
`rtol=1e-2`, and — the exact route — `scipy.integrate.quad` of
$v \cdot 4\pi v^2 (2\pi)^{-3/2} e^{-v^2/2}$ against the same closed form
at `rtol=1e-8`: sampling, formula, and quadrature all meeting on one
number.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    ratio_rel,
    np.sqrt(2.0),
    "the mean relative speed of two thermal molecules is √2 times the "
    "mean speed: the factor in every collision formula",
    rtol=1e-2,
)
validate.close(
    speed_mean,
    vbar_exact,
    "and the sampled mean speed lands on √(8kT/πm)",
    rtol=1e-2,
)
validate.close(
    vbar_quad,
    vbar_exact,
    "which quadrature of the Maxwell–Boltzmann speed density confirms "
    "to eight digits",
    rtol=1e-8,
)

## Exercise 2 — The numbers of the invisible world

Now {eq}`eq-kt-lambda` with real molecules. Nitrogen — four-fifths of
the room — has molecular mass $28.014\,\mathrm u$ and an effective
hard-sphere diameter $d = 0.375\ \mathrm{nm}$ (the standard
viscosity-derived value).

**Part a)** At $T = 300\ \mathrm K$ and $p = 1\ \mathrm{atm}$ compute,
with `scipy.constants` throughout: the number density $n = p/k_BT$, the
mean speed of {eq}`eq-kt-vrel`, the mean free path of
{eq}`eq-kt-lambda`, and the collision frequency $\nu = \langle v\rangle
/\lambda$. Verify $n = 2.45\times10^{25}\ \mathrm{m^{-3}}$,
$\langle v\rangle = 476\ \mathrm{m/s}$, $\lambda = 65.4\ \mathrm{nm}$,
and $\nu = 7.3\times10^{9}\ \mathrm{s^{-1}}$, each at `rtol=1e-2`: a
jetliner's speed, a virus's width of elbow room, and seven billion
collisions every second.

**Part b)** Put $\lambda$ on the ladder of lengths: verify
$\lambda / d = 174$ (dilute: a molecule flies 174 diameters between
hits) and $\lambda / n^{-1/3} = 19$ (the free path spans about nineteen
intermolecular spacings), both at `rtol=2e-2`. Then verify the
pressure law by recomputing at $p = 10^{-2}\ \mathrm{atm}$: $\lambda$
grows a hundredfold ($6.5\ \mathrm{\mu m}$, `rtol=1e-2` against
$100\times$ the atmospheric value) — the reason vacuum equipment cares
about kinetic theory.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([n_atm, vbar_n2, lam_n2 * 1e9, nu_n2]),
    np.array([2.45e25, 476.0, 65.4, 7.3e9]),
    "nitrogen at 300 K and 1 atm: 2.45e25 molecules/m³ at 476 m/s, "
    "65.4 nm of freedom, 7.3 billion collisions per second",
    rtol=1e-2,
)
validate.close(
    np.array([lam_n2 / D_N2, lam_n2 * n_atm ** (1 / 3)]),
    np.array([174.0, 19.0]),
    "the free path spans 174 diameters and 19 intermolecular spacings: "
    "dilute, but not empty",
    rtol=2e-2,
)
validate.close(
    lam_low,
    100.0 * lam_n2,
    "and at a hundredth of an atmosphere the free path grows exactly a "
    "hundredfold: λ ∝ 1/p",
    rtol=1e-2,
)

## Exercise 3 — The hard-disk laboratory

Formulas certified, we build the gas. An *event-driven* simulation
advances hard disks exactly: between collisions everything moves in
straight lines, so the next event — the earliest of all pair and wall
collisions — can be computed in closed form and executed with no
timestep error at all ({numref}`fig-kt-impact` shows the collision
condition). For a pair with relative position $\Delta\mathbf r$ and
relative velocity $\Delta\mathbf v$, contact
$|\Delta\mathbf r + \Delta\mathbf v\,t| = d$ is a quadratic in $t$
whose earlier root (when real, approaching, and positive) is the
collision time; the elastic impulse is exchanged along the line of
centers. **Write this one yourself** — the implementation is the
lesson.

The dimensional subtlety of the theory section becomes the measured
point: in this two-dimensional gas the collision cross-section is the
impact-parameter *segment* of width $2d$, so the prediction is
$\lambda_{\rm 2D} = 1/(2\sqrt2\,n\,d)$ with $n = N/L^2$.

**Part a)** Run $N = 120$ disks of diameter $d = 0.02$ in the unit box
($n = 120$, area fraction $3.8\%$) for $6000$ events from seeded
Maxwell–Boltzmann velocities, recording every free path (speed times
time since that disk's previous pair collision). Verify the mean free
path lands on $1/(2\sqrt2\,n\,d) = 0.147$ within $12\%$ (the residual
is the finite-density Enskog correction, a few percent at this area
fraction, plus statistics), and verify the free-path histogram is
exponential: the maximum deviation between the sorted paths' empirical
CDF and $1 - e^{-\ell/\bar\ell}$ (a Kolmogorov–Smirnov distance by
`numpy.sort` and direct comparison) below $0.06$.

**Part b)** Deliberately test the wrong formula: verify the same
measurement *rejects* $1/(\sqrt2\,n\,d)$ — the 3D-style expression with
the 2D width forgotten — by a factor of $2$ (measured/wrong below
$0.6$). A simulation that can falsify a mis-derived formula is worth
more than one that only confirms.

**Part c)** Animate $200$ snapshots of the gas: disks drifting in
straight lines, kinks appearing only at collisions. The animation's
physics is certified by Part a)'s free-path statistics, measured from
the same event sequence the frames sample.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    lam_meas,
    lam_2d,
    "the event-driven gas's mean free path lands on the 2D prediction "
    "1/(2√2 n d), Enskog-plus-statistics residual and all",
    rtol=0.12,
)
validate.check(
    ks_dist < 0.06,
    "and the full free-path distribution is exponential: uncorrelated "
    "collisions, as the Poisson picture demands",
    f"KS distance {ks_dist:.3f}",
)
validate.check(
    lam_meas / lam_wrong < 0.6,
    "the same data rejects the dimensionally careless formula (3D "
    "cross-section logic in 2D) by its factor of 2",
    f"measured/wrong = {lam_meas / lam_wrong:.3f}",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — Equipartition, enforced one collision at a time

[§5.4](microstates-entropy-temperature.ipynb) *derived* equipartition
from counting; here we watch the mechanism that enforces it. Load the
box with two species — half the disks at mass $m = 1$, half at $m = 4$
— and start the heavy species *cold*: at a quarter of the light
species' velocity scale, its mean kinetic energy per particle is
$4 \times (1/4)^2 = 1/4$ of the light one's. (Beware the tempting
"halve the velocities" version: for a mass-4 species the factor $4$
cancels $(1/2)^2$ exactly and the two species would *start* in
equipartition — a trap this exercise's first draft walked straight
into, and the energy-trace figure exposed.) Nothing couples the
species except collisions.

**Part a)** Run the two-species gas for $8000$ events, recording each
species' mean kinetic energy every $50$ events. Verify the initial
imbalance is real (heavy/light energy ratio below $0.5$ at the start)
and that collisions erase it: over the final quarter of the trace the
two means agree within $12\%$. Temperature, operationally, is the
thing hard collisions equalise.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    ratio_start < 0.5,
    "the two species genuinely start out of equilibrium: the heavy one "
    "at a quarter of the light one's mean energy",
    f"initial heavy/light energy ratio {ratio_start:.2f}",
)
validate.close(
    ratio_end,
    1.0,
    "and collisions alone drive them to equipartition: equal mean "
    "energies within the run's fluctuations",
    rtol=0.12,
)

## Exercise 5 — Transport: Maxwell's shock and Chapman–Enskog's percent

Now the payoff of {eq}`eq-kt-transport`. Argon is the clean test case
(monatomic, hard-sphere-like), with mass $39.948\,\mathrm u$ and
viscosity diameter $d = 0.364\ \mathrm{nm}$; the measured viscosity at
$300\ \mathrm K$ is $\eta_{\rm exp} = 22.7\ \mathrm{\mu Pa\,s}$.

**Part a)** Maxwell's cancellation. Evaluate the elementary
`eta_elementary` at $p = 0.1$, $1$, and $10\ \mathrm{atm}$ (density
$n = p/k_BT$ each time) and verify the three values are *identical* to
`rtol=1e-12`: the carriers double, their reach halves, and pressure
drops out exactly — the prediction Maxwell found absurd enough to test
himself {cite}`maxwell1867`.

**Part b)** The honest hierarchy. Verify the elementary estimate lands
at $15.0\ \mathrm{\mu Pa\,s}$ (`rtol=2e-2`): the right physics within a
factor $1.5$, which is what a mnemonic prefactor buys. Then verify the
Chapman–Enskog value of {eq}`eq-kt-enskog` lands on the measured
$22.7\ \mathrm{\mu Pa\,s}$ to `rtol=5e-2`: solving the Boltzmann
equation properly is worth exactly the missing $50\%$.

**Part c)** The same $\tfrac13\langle v\rangle\lambda$ logic prices
diffusion: verify $D = \tfrac13\langle v\rangle\lambda$ for nitrogen at
STP gives $1.0\times10^{-5}\ \mathrm{m^2/s}$ (`rtol=5e-2`) — the
centimetre-per-minute scale of gas mixing... and then note what it does
*not* explain: a scent crosses a still room in seconds, not the
$\sim$hours pure diffusion would need over metres. Verify the diffusion
time $t = L^2/(2D)$ for $L = 3\ \mathrm m$ exceeds $10^5\ \mathrm s$:
what actually carries the scent is convection, and the estimate proves
it by elimination.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    float(np.max(np.abs(etas_p / etas_p[1] - 1.0))) < 1e-12,
    "Maxwell's shock: the elementary viscosity is IDENTICAL at 0.1, 1, "
    "and 10 atm — pressure cancels exactly",
    f"spread {float(np.max(np.abs(etas_p / etas_p[1] - 1.0))):.1e}",
)
validate.close(
    eta_elem * 1e6,
    15.0,
    "the mnemonic (1/3) n m <v> λ estimate lands within a factor 1.5 of "
    "the measured argon viscosity",
    rtol=2e-2,
)
validate.close(
    eta_ce,
    ETA_AR_EXP,
    "and Chapman–Enskog's proper solution of the Boltzmann equation "
    "meets the measured 22.7 µPa·s at the percent level",
    rtol=5e-2,
)
validate.close(
    D_n2,
    1.0e-5,
    "the same logic prices nitrogen's self-diffusion at 1e-5 m²/s",
    rtol=5e-2,
)
validate.check(
    t_room_diff > 1e5,
    "so pure diffusion would need over a day to cross a room: the scent "
    "that reaches you in seconds rides convection, by elimination",
    f"t = {t_room_diff:.0f} s",
)

## Exercise 6 — Effusion: the hole as a mass filter

{numref}`fig-kt-effusion-schematic` shows the setup for
{eq}`eq-kt-effusion`: a hole smaller than $\lambda$, so escape is
single-molecule ballistics, no hydrodynamics.

In [ ]:
# (solution hidden on the public site)


**Part a)** The quarter. Sample $10^6$ Maxwell–Boltzmann $z$-velocity
components for argon at $300\ \mathrm K$ (Gaussian, scale
$\sqrt{k_BT/m}$, seeded) and form the escape flux factor: the mean of
$v_z$ over the *positive side only*, times the fraction moving toward
the hole, divided by $\langle v\rangle$. Verify it equals $1/4$ within
`rtol=1e-2`: the $\tfrac14 n\langle v\rangle$ of {eq}`eq-kt-effusion`,
by direct count.

**Part b)** The escaping beam is fast. Effused molecules are sampled
with probability $\propto v_z$, and for the isotropic Maxwell
distribution the flux-weighted mean speed works out to
$\langle v^2\rangle / \langle v\rangle$, so the ratio to the gas's mean
is exactly $\langle v^2\rangle/\langle v\rangle^2 = 3\pi/8 = 1.178$.
Weight the full 3D speed samples by their positive $v_z$ and verify the
sampled ratio against both the closed form $3\pi/8$ and the
`scipy.integrate.quad` evaluation of $\int v\cdot v f(v)\,dv \big/
\int v f(v)\,dv$ over the Maxwell speed density, each at `rtol=1e-2`.

**Part c)** Graham's law at war. For the uranium hexafluorides, masses
$349.03$ and $352.04\ \mathrm u$, verify the single-stage enrichment
$\alpha = \sqrt{352.04/349.03} = 1.00430$ (`rtol=1e-5`), then chain
ideal stages ($R \mapsto \alpha R$ in the abundance *ratio* $R = x/(1-
x)$ starting from natural $x = 0.72\%$): verify reactor-grade $3.6\%$
needs $382$ stages and weapons-grade $90\%$ needs $1660$ (exact integer
counts by `numpy.ceil` of the logarithm ratio). The wartime K-25 plant
ran about three thousand stages in cascade; a factor of $1.004$,
compounded with enough patience and electricity, moved history.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    flux_factor,
    0.25,
    "the escape flux is n<v>/4 by direct Monte Carlo count of "
    "hole-crossing molecules",
    rtol=1e-2,
)
validate.close(
    beam_ratio,
    3.0 * np.pi / 8.0,
    "the effused beam is faster than the gas it left by exactly 3π/8 = "
    "1.178: flux-weighting favours the quick",
    rtol=1e-2,
)
validate.close(
    beam_quad,
    3.0 * np.pi / 8.0,
    "as the quadrature of the flux-weighted speed density confirms",
    rtol=1e-6,
)
validate.close(
    alpha_g,
    1.00430,
    "Graham's law gives UF6 a single-stage enrichment of 1.0043",
    rtol=1e-5,
)
validate.check(
    stages_leu == 382 and stages_heu == 1660,
    "and compounding it demands 382 ideal stages for reactor grade, "
    "1660 for weapons grade: patience as a separation technology",
    f"{stages_leu} and {stages_heu} stages",
)

## Notebook summary

- The $\sqrt 2$ of every collision formula was measured
  ($\langle v_{\rm rel}\rangle/\langle v\rangle = 1.411$ from
  $2\times10^5$ sampled pairs) and the mean speed met its closed form
  by sampling and by quadrature.
- Nitrogen at room conditions carries the canonical numbers: $\lambda =
  65.4\ \mathrm{nm}$ (174 diameters, 19 spacings), $\langle v\rangle =
  476\ \mathrm{m/s}$, $\nu = 7.3\times10^9\ \mathrm{s^{-1}}$, with
  $\lambda \propto 1/p$ verified across two decades of pressure.
- The event-driven hard-disk gas — exact dynamics, no timestep —
  delivered exponential free paths (KS distance $0.035$) whose mean
  landed on the two-dimensional $1/(2\sqrt2\,nd)$ and *rejected* the
  dimensionally careless formula by its factor of $2$; loaded with two
  species at unequal temperatures, its collisions alone enforced
  equipartition.
- Transport followed from $\tfrac13\langle v\rangle\lambda$: Maxwell's
  pressure-independence of viscosity held to $10^{-12}$, the elementary
  argon estimate landed within a factor $1.5$ of the measured
  $22.7\ \mathrm{\mu Pa\,s}$, and Chapman–Enskog closed the rest to
  percent level; nitrogen's $D \approx 10^{-5}\ \mathrm{m^2/s}$ proved
  by elimination that scents cross rooms by convection.
- Effusion delivered its quarter ($\Phi = n\langle v\rangle/4$ by
  direct count), its fast beam ($1.18\times$), and Graham's law at its
  most consequential: $\alpha = 1.0043$ per stage, $382$ stages to
  reactor grade, $1660$ to weapons grade.

## Outlook

- **The Boltzmann equation.** Everything here is its shadow: the full
  equation evolves the one-particle distribution $f(\mathbf r, \mathbf
  v, t)$ under streaming and a bilinear collision integral, and
  Chapman–Enskog {cite}`chapmancowling1970` extracts hydrodynamics —
  Navier–Stokes with *computable* coefficients — as its
  small-gradient limit. The H-theorem face of it appeared in
  [§5.11](taste-of-nonequilibrium.ipynb).
- **Rarefied gases.** When $\lambda$ reaches the apparatus size (the
  Knudsen regime), the hydrodynamic limit fails and effusion-style
  ballistics takes over: vacuum technology, atmospheric re-entry, and
  microfluidics all live there.
- **From hard disks to molecular dynamics.** The event-driven gas is
  the ancestor of the molecular-dynamics simulations toward which this
  volume has been pointing; swap the hard core for the smooth
  potentials of [MMM's](https://ramador09.github.io/molecular-materials-modelling-public/) world and the velocity Verlet of
  [§1.6](../01-elementary-mechanics/integrators.ipynb) takes over from
  exact event scheduling.
- **Isotopes after the war.** Gaseous diffusion gave way to
  centrifuges (whose separation factor rides $\Delta m$ rather than
  $\sqrt{m_2/m_1}$, a far better deal for heavy elements) — kinetic
  theory still, one clever geometry later.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()